In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 66 — Calendar Scheduling Agent

## Goal
Build an agent that can **propose meeting times**, handle
**scheduling conflicts**, and **send follow-up summaries**.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Calendar tools** | Check availability, create/cancel events |
| **Conflict resolution** | Detect overlaps and suggest alternatives |
| **Structured scheduling** | Parse natural language into time slots |
| **Follow-up generation** | Summarize meeting and draft follow-up |

## Architecture
```
"Schedule a meeting with Alice and Bob next Tuesday"
  → check_calendar → find_free_slots → propose_meeting → confirm
```

## Stack
- `LangGraph` + `ChatOllama`
- Simulated calendar system (in-memory)

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from datetime import datetime, timedelta
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

print("All imports OK")

## Step 1 — Simulated Calendar System

We create an in-memory calendar with existing events
for multiple people.

In [ ]:
# Simulated calendar database
BASE_DATE = datetime(2025, 6, 16)  # A Monday

CALENDARS = {
    "alice": [
        {"title": "Team Standup", "start": "2025-06-16 09:00", "end": "2025-06-16 09:30", "recurring": "daily"},
        {"title": "1:1 with Manager", "start": "2025-06-16 14:00", "end": "2025-06-16 14:30", "recurring": None},
        {"title": "Sprint Planning", "start": "2025-06-17 10:00", "end": "2025-06-17 11:30", "recurring": None},
        {"title": "Lunch", "start": "2025-06-16 12:00", "end": "2025-06-16 13:00", "recurring": "daily"},
    ],
    "bob": [
        {"title": "Team Standup", "start": "2025-06-16 09:00", "end": "2025-06-16 09:30", "recurring": "daily"},
        {"title": "Client Call", "start": "2025-06-16 11:00", "end": "2025-06-16 12:00", "recurring": None},
        {"title": "Design Review", "start": "2025-06-17 14:00", "end": "2025-06-17 15:00", "recurring": None},
        {"title": "Lunch", "start": "2025-06-16 12:00", "end": "2025-06-16 13:00", "recurring": "daily"},
    ],
    "carol": [
        {"title": "Team Standup", "start": "2025-06-16 09:00", "end": "2025-06-16 09:30", "recurring": "daily"},
        {"title": "All Hands", "start": "2025-06-16 15:00", "end": "2025-06-16 16:00", "recurring": None},
        {"title": "Lunch", "start": "2025-06-16 12:00", "end": "2025-06-16 13:00", "recurring": "daily"},
    ],
}

_created_events = []

print("Calendar system initialized")
for person, events in CALENDARS.items():
    print(f"  {person}: {len(events)} events")

## Step 2 — Define Calendar Tools

In [ ]:
@tool
def check_calendar(person: str, date: str) -> str:
    """Check a person's calendar for a given date (YYYY-MM-DD format). Returns all events for that day."""
    person = person.lower()
    if person not in CALENDARS:
        return f"No calendar found for {person}. Available: {list(CALENDARS.keys())}"
    events = CALENDARS[person]
    day_events = [e for e in events if e["start"].startswith(date)]
    # Include recurring daily events
    for e in events:
        if e["recurring"] == "daily" and e not in day_events:
            adj = {**e, "start": date + e["start"][10:], "end": date + e["end"][10:]}
            day_events.append(adj)
    if not day_events:
        return f"{person} has no events on {date} — fully available."
    lines = [f"{person}'s schedule for {date}:"]
    for e in sorted(day_events, key=lambda x: x["start"]):
        lines.append(f"  {e['start'][11:]} - {e['end'][11:]}  {e['title']}")
    return "\n".join(lines)

@tool
def find_free_slots(people: str, date: str, duration_minutes: int) -> str:
    """Find common free time slots for multiple people (comma-separated names) on a date. Working hours: 09:00-17:00."""
    names = [n.strip().lower() for n in people.split(",")]
    missing = [n for n in names if n not in CALENDARS]
    if missing:
        return f"No calendar for: {missing}. Available: {list(CALENDARS.keys())}"
    
    # Build busy intervals
    busy = []
    for name in names:
        for e in CALENDARS[name]:
            event_date = e["start"][:10]
            if event_date == date or e["recurring"] == "daily":
                start_h, start_m = map(int, e["start"][11:].split(":"))
                end_h, end_m = map(int, e["end"][11:].split(":"))
                busy.append((start_h * 60 + start_m, end_h * 60 + end_m))
    
    busy.sort()
    # Find gaps in working hours (9:00 = 540 to 17:00 = 1020)
    free = []
    cursor = 540  # 9:00
    for start, end in busy:
        if start > cursor and start - cursor >= duration_minutes:
            free.append((cursor, start))
        cursor = max(cursor, end)
    if 1020 - cursor >= duration_minutes:
        free.append((cursor, 1020))
    
    if not free:
        return f"No {duration_minutes}-minute slots available for {', '.join(names)} on {date}."
    
    lines = [f"Available {duration_minutes}-min slots for {', '.join(names)} on {date}:"]
    for start, end in free:
        lines.append(f"  {start//60:02d}:{start%60:02d} - {end//60:02d}:{end%60:02d}")
    return "\n".join(lines)

@tool
def create_event(title: str, people: str, start_time: str, end_time: str) -> str:
    """Create a calendar event. people: comma-separated names. start_time/end_time: 'YYYY-MM-DD HH:MM' format."""
    names = [n.strip().lower() for n in people.split(",")]
    event = {
        "title": title,
        "start": start_time,
        "end": end_time,
        "attendees": names,
        "recurring": None,
    }
    _created_events.append(event)
    for name in names:
        if name in CALENDARS:
            CALENDARS[name].append(event)
    return f"✅ Event created: '{title}' with {', '.join(names)} from {start_time} to {end_time}"

@tool
def draft_meeting_summary(title: str, attendees: str, key_points: str) -> str:
    """Draft a meeting follow-up summary email. key_points: semicolon-separated list of discussion points."""
    points = [p.strip() for p in key_points.split(";")]
    summary = f"""Subject: Meeting Summary — {title}

Hi {attendees},

Thank you for attending today's meeting. Here's a summary:

Key Points:
"""
    for i, point in enumerate(points, 1):
        summary += f"  {i}. {point}\n"
    summary += "\nPlease reach out if you have any questions or action items.\n\nBest regards,\nScheduling Assistant"
    return summary

cal_tools = [check_calendar, find_free_slots, create_event, draft_meeting_summary]
print(f"Defined {len(cal_tools)} calendar tools")

## Step 3 — Create the Calendar Agent

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0)

CAL_PROMPT = """You are a calendar scheduling assistant. Today is Monday, June 16, 2025.

You help people schedule meetings by:
1. Checking everyone's calendars
2. Finding common free time slots
3. Proposing the best time
4. Creating the event
5. Drafting follow-up summaries when asked

Rules:
- Working hours are 9:00 AM to 5:00 PM
- Respect lunch breaks (12:00-13:00)
- Prefer morning slots over afternoon when possible
- Always check for conflicts before creating events
- Suggest alternatives if the requested time doesn't work"""

cal_agent = create_react_agent(
    model=llm,
    tools=cal_tools,
    prompt=CAL_PROMPT,
)

def ask_scheduler(request: str):
    print(f"\n{'='*70}")
    print(f"REQUEST: {request}")
    print(f"{'='*70}")
    result = cal_agent.invoke({"messages": [{"role": "user", "content": request}]})
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🔧 TOOL: {tc['name']}({str(tc['args'])[:120]})")
        elif role == "ToolMessage":
            print(f"📋 RESULT: {msg.content[:400]}")
        elif role == "AIMessage" and msg.content:
            print(f"\n🤖 ASSISTANT:\n{msg.content}")
    return result

print("Calendar agent ready!")

In [ ]:
# Task 1: Simple scheduling
ask_scheduler("Schedule a 30-minute meeting with Alice and Bob on June 16. Find a time that works for both.")

In [ ]:
# Task 2: Three-person meeting
ask_scheduler("I need a 1-hour meeting with Alice, Bob, and Carol on Tuesday June 17. When can we all meet?")

In [ ]:
# Task 3: Follow-up summary
ask_scheduler("Draft a follow-up summary for the Sprint Planning meeting with Alice and Bob. Key points: reviewed Q2 roadmap; assigned feature X to Alice; deadline is July 15; next sprint starts Monday.")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Calendar check** | Query per-person event lists |
| **Slot finding** | Find gaps in merged busy intervals |
| **Conflict handling** | Detect overlaps before creating events |
| **Follow-up drafts** | Structured summary from key points |

## Real-World Integration
```
Replace in-memory calendar with:
- Google Calendar API (via MCP or OAuth)
- Microsoft Graph API (Outlook)
- CalDAV (open standard)

Add features:
- Timezone handling
- Recurring event creation
- Room booking
- Auto-send invites via email
```